# 4. Analysis using topological features
This notebook illustrates the key analysis that are performed on the topological features. 

Note that the analysis is done on a larger collection of dataset.

In this notebook, we illustrate the process of performing UMAP and PCA on the PH feature vectors extracted from ECM point cloud. One could perform similar analysis on dim-0 and dim-1 PH feature vectors obtained from ECM, cancer cells, leukocytes, and dim-0 and dim-1 Dowker PH features, and any combinations of the feature vectors


In [ ]:
using Pkg
Pkg.activate("../.")
Pkg.instantiate()

In [ ]:
include("../src/ECM_TDA.jl")
using .ECM_TDA

using JLD2
using Statistics
using UMAP
using Plots
using Plots.PlotMeasures
using DataFrames
gr()

In [ ]:
function plot_PI3(PI, x_min, x_max, y_min, y_max; 
    kwargs...) 

    n = size(PI,1)
    xinterval = (x_max - x_min)/4
    yinterval = (y_max - y_min)/4
    p = heatmap(PI, 
    label = "",
    title = "",
    framestyle = :box,
    xticks = (5:5:20, Int32.(round.(x_min+xinterval:xinterval:x_max))),
    yticks = (5:5:20, Int32.(round.(y_min+yinterval:yinterval:y_max))),
    ;kwargs...)

    return p
end

# Analysis on ECM point cloud

## 1(a) combine the dim-0 and dim-1 persistent homology features

In [ ]:
# load the persistence images 
ECM_PI = load("persistent_homology_outputs/ECM/PI.jld2")
ECM_PI0 = ECM_PI["PI0_ECM"]
ECM_PI1 = ECM_PI["PI1_ECM"];

In [ ]:
# combine features to get a 420-dimensional vector for each ROI 
features = Dict()
for f in keys(ECM_PI0)
    # check that f is a key in all dictionaries
    if f in keys(ECM_PI1)
        combined = vcat(ECM_PI0[f], vec(ECM_PI1[f]))
        features[f] = combined
    end
end

### mean-center the features
# collect the ROI names, and assign a number
ROIs = collect(keys(features))
idx_ROI = Dict(i => roi for (i, roi) in enumerate(ROIs));

n = length(idx_ROI)
features_array = hcat([features[idx_ROI[i]] for i = 1:n]...)
println("features array shape: ", size(features_array))

features_centered = features_array .- mean(features_array, dims = 2);

## 1(b) perform UMAP

Here, we perform UMAP on the combined features. One can perform UMAP on the dim-0 or dim-1 persistent homology features separately.

In [ ]:
# perform UMAP & save
embedding = umap(features_centered, 2; n_neighbors = 5)

n = size(embedding, 2)
p = scatter(embedding[1,:], embedding[2,:], 
        markercolor = "slategrey",
        markersize = 5, 
        label = "", 
        xticks = [], 
        yticks = [], 
        framestyle = :box,  
        xlabel = "UMAP-1",
        ylabel = "UMAP-2",
        guidefontsize = 15,
        leftmargin = 5mm,
        size = (450, 350),
        legend = :topright)

## 1(c) perform PCA

Here, we perform PCA on dim-1 persistent homology features.

Note that the analysis in the manuscript is performed on a much larger dataset. Therefore, the principal components in this notebook may represent very different information than the principal components in the manuscript. 

In [ ]:
# convert PI1 to a dictionary indexed by numbers according to "idx_ROI"
PI1_new = Dict(i => ECM_PI1[idx_ROI[i]] for i = 1:length(idx_ROI))

In [ ]:
# compute PCA
transformed, eigenvectors, variance_1, variance_2, _ = PI_to_PCA(PI1_new; pratio = 0.99)


println("number of components: ", length(eigenvectors))
println("variance explained by 1 eigenvectors: ", variance_1)
println("variance explained by 2 eigenvectors: ", variance_2)
println("variance difference between 2 and 1:", variance_2 - variance_1)

In [ ]:
# save results of PCA to a dataframe
col_names = ["pca_coord_" * string(i) for i = 1:6];
df_pca1 = DataFrame(Array(transpose(transformed)), col_names)
idx_ROI_list = [idx_ROI[i] for i = 1:length(idx_ROI)];
df_pca1[:, :idx_ROI] = idx_ROI_list;

In [ ]:
PC1 = Int(round(100*variance_1,digits=0))
PC2 = Int(round(100*(variance_2-variance_1),digits=0))

p = scatter(df_pca1[:,"pca_coord_1"], df_pca1[:,"pca_coord_2"], 
        markercolor = "lightgray",
        alpha = 1,
        markersize = 5, 
        markerstrokewidth = 3,
        label = "", 
        xaxis = "PC1 ($PC1%)",
        yaxis = "PC2 ($PC2%)",  
        xticks = (0, 0),
        yticks = (0,0),
        guidefontsize = 7,
        framestyle = :box,
        size = (200, 150),
        legend = :bottomleft
        )
plot(p)

## Plot the eigenvectors of PCA as persistence images

In [ ]:
# get min and max pixels of the first four eigenvectors
eigenvector_min = minimum(minimum.(eigenvectors[i] for i = 1:4))
eigenvector_max = maximum(maximum.(eigenvectors[i] for i = 1:4))

println("min pixel: ", eigenvector_min)
println("max pixel: ", eigenvector_max)

In [ ]:
# for plotting purposes, need to know the x- and y-axis min and max values of the persitence images 

PI_ranges = load("persistent_homology_outputs/ECM/PI_ranges.jld2")

PI0_xmin = PI_ranges["PI0_xmin"]
PI0_xmax = PI_ranges["PI0_xmax"]
PI0_ymin = PI_ranges["PI0_ymin"]
PI0_ymax = PI_ranges["PI0_ymax"]
PI1_xmin = PI_ranges["PI1_xmin"]
PI1_xmax = PI_ranges["PI1_xmax"]
PI1_ymin = PI_ranges["PI1_ymin"]
PI1_ymax = PI_ranges["PI1_ymax"]

In [ ]:
#plot the first two eigenvectors 

plot_scale = 20 # only show plot_scale% of persistence image
ps = [plot_PI2(eigenvectors[i], PI1_xmin, PI1_xmax, PI1_ymin, PI1_ymax, 
            clims = (eigenvector_min, eigenvector_max), 
            xlabel = "birth",
            ylabel = "persistence",
            show_axis = false,
            left_margin = 5mm,
            bottom_margin = 5mm,
            xrotation = 45,
            #framestyle = :box,
            #title = "eigenvector " * string(i), 
            legend = :false # no colorbar 
            ) for i =1:2]

l = @layout[grid(1,2) a{0.05w}] # Stack a layout that rightmost one is for color bar
Plots.GridLayout(1, 2)

n = 100 # length of colorbar (as a vector)
cbar_interval = 0.2
cbar_ticks = vcat(reverse(collect(0:cbar_interval: -eigenvector_min))[1:end-1] .* -1, collect(0:cbar_interval:eigenvector_max))
cbar_loc = [cbar_tickvals_to_loc(eigenvector_min, eigenvector_max, n, val) for val in cbar_ticks]

p = plot(ps..., 
         heatmap(collect(range(eigenvector_min, eigenvector_max, length = n)) .* ones(n,1), 
                legend=:none, 
                xticks=:none,
                yticks=(cbar_loc, cbar_ticks)),
         layout=l,
         topmargin = 3mm,
         size = (500, 200))
plot(p)